In [ ]:
# import pandas as pd
# import numpy as np


# def load_dataset(name_dataset):
#     #leemos el dataset de futbol uruguayo
#     df = pd.read_csv(name_dataset)

#     #nos quedamos con las columnas que nos interesan para el clasificador
#     df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

#     #convertimos las columnas a los tipos de datos correctos
#     df["date"] = pd.to_datetime(df["date"], errors="raise")
#     df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
#     df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

#     #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
#     df = (
#         df.drop_duplicates()
#         .sort_values(["date", "home", "away"], kind="stable")
#         .reset_index(drop=True)
#     )

#     #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
#     df["result"] = df.apply(
#         lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
#     )

#     return df

# def load_atributes(dataset: pd.DataFrame, date: pd.Timestamp, years_limit: int = 1, matches_limit: int = 5) -> pd.DataFrame:

#     def get_historial(date, team: str) -> pd.DataFrame:

#         #filtramos el dataset por el equipo y la fecha
#         df = dataset[(dataset["home"] == team) | (dataset["away"] == team)]
#         df = df[(date - pd.DateOffset(years=years_limit) < df["date"]) & (df["date"] < date)]

#         #eliminamos los que sean anteriores a un limite 
#         return df

#     def get_wins(dataset: pd.DataFrame, team: str) -> int:
#         #contamos las victorias del equipo
#         return len(dataset[(dataset["home"] == team) & (dataset["result"] == "L")]) + len(dataset[(dataset["away"] == team) & (dataset["result"] == "V")])

#     def comparar(win_reate_team1: float, win_reate_team2: float) -> str:
#         #comparamos las victorias de los equipos
#         if win_reate_team1 > win_reate_team2:
#             return "L"
#         elif win_reate_team1 < win_reate_team2:
#             return "V"
#         else:
#             return "E"

#     def get_condition(date, team: str) -> int:
#         #obtenemos el historial del equipo
#         df = get_historial(date, team)

#         #me quedo con los ultimos partidos del equipo (si los hay)
#         df = df.tail(matches_limit)

#         #TODO : quizas se puede hacer por puntos en vez de cantidad de victorias, pero por ahora lo dejamos asi
#         return (get_wins(df, team) / len(df) if len(df) > 0 else 0)
    

#     #creamos las nuevas columnas con el historial historico de los equipos
#     dataset["historial"] = comparar(dataset["home"].apply(get_historial, args=(dataset["date"], dataset["home"])), dataset["away"].apply(get_historial, args=(dataset["date"], dataset["away"])))

#     #creamos las nuevas columnas con las condiciones recientes de los equipos
#     dataset["condition_match"] = comparar(dataset["home"].apply(get_condition, args=(dataset["date"], dataset["home"])), dataset["away"].apply(get_condition, args=(dataset["date"], dataset["away"])))  
    
#     return dataset

    







In [1]:
import pandas as pd
import numpy as np

def load_dataset(name_dataset):
    #leemos el dataset de futbol uruguayo
    df = pd.read_csv(name_dataset)

    #nos quedamos con las columnas que nos interesan para el clasificador
    df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

    #convertimos las columnas a los tipos de datos correctos
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
    df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

    #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
    df = (
        df.drop_duplicates()
        .sort_values(["date", "home", "away"], kind="stable")
        .reset_index(drop=True)
    )

    #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
    df["result"] = df.apply(
        lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
    )

    return df

#funcion que se usa para generar nuevos atributos a partir del dataset origial
def load_attributes(
    dataset: pd.DataFrame,
    years_limit: int = 1,
    matches_limit: int = 5
) -> pd.DataFrame:

    dataset = dataset.copy()

    #funcion que obtiene el historial de partidos de un equipo hasta una fecha determinada
    def get_record(
        date: pd.Timestamp,
        team: str
    ) -> pd.DataFrame:

        start_date = date - pd.DateOffset(
            years=years_limit
        )

        record = dataset[
            (
                (dataset["home"] == team)
                | (dataset["away"] == team)
            )
            & (dataset["date"] >= start_date)
            & (dataset["date"] < date)
        ]

        return record.sort_values("date")

    #funcion que obtiene la cantidad de victorias de un equipo en un historial de partidos
    def get_wins(
        record: pd.DataFrame,
        team: str
    ) -> int:

        home_victories = (
            (record["home"] == team)
            & (record["result"] == "L")
        ).sum()

        away_victories = (
            (record["away"] == team)
            & (record["result"] == "V")
        ).sum()

        return int(
            home_victories + away_victories
        )

    #funcion que obtiene la tasa de victorias de un equipo en un historial de partidos
    def get_win_rate(
        record: pd.DataFrame,
        team: str
    ) -> float:

        if len(record) == 0:
            return 0.0

        return get_wins(record, team) / len(record)

    #funcion que obtiene la tasa de puntos de un equipo en un historial de partidos
    def get_points_rate(
        record: pd.DataFrame,
        team: str
    ) -> float:

        if len(record) == 0:
            return 0.0

        home_points = (
            (record["home"] == team)
            & (record["result"] == "L")
        ).sum() * 3 + (
            (record["home"] == team)
            & (record["result"] == "E")
        ).sum()

        away_points = (
            (record["away"] == team)
            & (record["result"] == "V")
        ).sum() * 3 + (
            (record["away"] == team)
            & (record["result"] == "E")
        ).sum()

        return (home_points + away_points) / (len(record) * 3)

    #funcion que obtiene la diferencia de goles de un equipo en un historial de partidos
    def goal_difference_local(
        record: pd.DataFrame,
        team: str
    ) -> float:
        local_matches = record[
            record["home"] == team
        ]

        if len(local_matches) == 0:
            return 0.0

        goals_for = local_matches["gh"].mean()
        goals_against = local_matches["ga"].mean()

        return float(goals_for - goals_against)

    #funcion que obtiene la diferencia de goles de un equipo en un historial de partidos
    def goal_difference_away(
        record: pd.DataFrame,
        team: str
    ) -> float:
        away_matches = record[
            record["away"] == team
        ]

        if len(away_matches) == 0:
            return 0.0

        goals_for = away_matches["ga"].mean()
        goals_against = away_matches["gh"].mean()

        return float(goals_for - goals_against)

    #funcion que compara dos tasas y devuelve "L" si la primera es mayor, "V" si la segunda es mayor y "E" si son iguales
    def compare(
        home_rate: float,
        away_rate: float
    ) -> str:

        if home_rate > away_rate:
            return "L"
        elif home_rate < away_rate:
            return "V"
        return "E"

    #funcion que devuelve un nivel de experiencia en base a la cantidad de partidos jugados
    def level_experience(quantity: int) -> int:
        if quantity == 0:
            return 0  # sin información
        elif quantity < 3:
            return 1  # poca información
        elif quantity < 5:
            return 2  # información intermedia
        else:
            return 3  # información suficiente

    #funcion que calcula los nuevos atributos para un partido dado
    def calculate_attributes(
        row: pd.Series
    ) -> pd.Series:

        date = row["date"]
        home_team = row["home"]
        away_team = row["away"]

        home_record = get_record(
            date,
            home_team
        )
        away_record = get_record(
            date,
            away_team
        )

        home_rate = get_win_rate(
            home_record,
            home_team
        )
        away_rate = get_win_rate(
            away_record,
            away_team
        )

        ventaja_historica = compare(
            home_rate,
            away_rate
        )

        home_lasts = home_record.tail(
            matches_limit
        )
        away_lasts = away_record.tail(
            matches_limit
        )

        home_condition = get_points_rate(
            home_lasts,
            home_team
        )
        away_condition = get_points_rate(
            away_lasts,
            away_team
        )

        last_matches = compare(
            home_condition,
            away_condition
        )

        goal_difference = compare(
            goal_difference_local(home_record, home_team),
            goal_difference_away(away_record, away_team)
        )

        local_experience = level_experience(
            len(home_record)
        )

        away_experience = level_experience(
            len(away_record)
        )

        record_enough = int(
            len(home_record) >= matches_limit
            and len(away_record) >= matches_limit
        )

        return pd.Series({
            "record": ventaja_historica,
            "last_matches": last_matches,
            "goal_difference": goal_difference,
            "local_experience": local_experience,
            "away_experience": away_experience,
            "record_enough": record_enough,
        })

    new_attributes = dataset.apply(
        calculate_attributes,
        axis=1
    )

    return pd.concat(
        [dataset, new_attributes],
        axis=1
    )

In [2]:
from sklearn.preprocessing import OrdinalEncoder


df = load_dataset("futbol_uruguayo.csv")

df_procesado = load_attributes(
    df,
    years_limit=1,
    matches_limit=5
)


df_procesado = df_procesado.drop(columns=["date"])
cols_procesadas=["home", "away", "record", "last_matches", "goal_difference", "local_experience", "away_experience", "record_enough", "result"]  

enc = OrdinalEncoder(dtype=int)

# fit_transform aprende las categorías y transforma todo el bloque
df_procesado[cols_procesadas] = enc.fit_transform(df_procesado[cols_procesadas])

# for i, cat in enumerate(enc.categories_[0]):
#     print(f"{cat} -> {i}")

print(
    df_procesado[
        [
            #"date",
            "home",
            "away",
            "record",
            "last_matches",
            "goal_difference",
            "local_experience",
            "away_experience",
            "record_enough",
            "result"
        ]
    ].head(20)
)

    home  away  record  last_matches  goal_difference  local_experience  \
0      1    17       0             0                0                 0   
1      7    33       0             0                0                 0   
2     10    31       0             0                0                 0   
3     26    30       0             0                0                 0   
4     27    22       0             0                0                 0   
5      7    22       0             1                1                 1   
6     17    30       1             1                1                 1   
7     26    10       0             0                1                 1   
8     27    33       1             1                1                 1   
9     31     1       0             0                0                 1   
10     7    10       0             0                1                 1   
11    17    31       1             1                1                 1   
12    26    22       1   

In [3]:
for i, cat in enumerate(enc.categories_[0]):
    print(f"{cat} -> {i}")

Albion -> 0
Bella Vista -> 1
Boston River -> 2
CA Basanez -> 3
CA Cerro -> 4
CA Fenix -> 5
CA Juventud -> 6
CA Penarol -> 7
CA Progreso -> 8
CSyd Villa Espanola -> 9
Central Espanol -> 10
Cerro Largo FC -> 11
Club Atletico Atenas -> 12
Club Social Y Deportivo Huracan Buceo -> 13
Club Sportivo Cerrito -> 14
Colon FC -> 15
Danubio -> 16
Defensor Sporting -> 17
Dep Colonia -> 18
Deportivo Maldonado -> 19
El Tanque Sisley -> 20
Frontera Rivera Chico -> 21
Institucion Atletica Sud America -> 22
La Luz Football Club -> 23
Liverpool -> 24
Miramar Misiones -> 25
Montevideo Wanderers -> 26
Nacional -> 27
Paysandu FC -> 28
Plaza Colonia -> 29
Racing Club -> 30
Rampla Juniors Futbol Club -> 31
Rentistas -> 32
River Plate -> 33
Rocha Futbol Club -> 34
Tacuarembo Futbol Club -> 35
Torque FC -> 36
Villa Teresa -> 37


In [7]:
import sys
import os

# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))

# Ahora ya puedes importar el archivo o sus elementos
from decisionTree.clasifier import clasifier as DecisionTreeClassifier
# o importar funciones/clases específicas:
# from decisionTree.clasifier import MiClase, mi_funcion

#esta es la carga del dataset de pruebas
#dataset=pd.read_csv("../../datos_procesados/entrenamiento_codificado.csv")

modelo = DecisionTreeClassifier(0.003)

atributos = ["record",
            "last_matches",
            "goal_difference",
            "local_experience",
            "away_experience",
            "record_enough"]
#atributos= ["ventaja_historica","ventaja_forma","ataque_local","defensa_local","ataque_visitante","defensa_visitante","ventaja_localia"]
# print(df_procesado.isna().sum())
# print(df_procesado.dtypes)
# print(df_procesado[df_procesado.isna().any(axis=1)].head())
# print(df_procesado.head())


modelo.fit(atributos, df_procesado)
modelo.tree.print_tree()




record
  [0]
    goal_difference
      [0]
        last_matches
          [0]
            away_experience
              [0]
                1
              [1]
                1
              [3]
                1
          [2]
            1
          [1]
            record_enough
              [1]
                0
              [0]
                2
      [1]
        record_enough
          [0]
            last_matches
              [1]
                away_experience
                  [1]
                    1
                  [2]
                    1
                  [3]
                    local_experience
                      [1]
                        1
                      [2]
                        0
              [0]
                away_experience
                  [1]
                    0
                  [2]
                    1
              [2]
                0
          [1]
            last_matches
              [1]
                1
              [0]
       